# GVA2035_2_모형추정

시나리오별 산업 실질부가가치 전망(2025~2035) 개발 코드. API 키(API_KEY_BOK.txt, API_KEY_KOSIS.txt) 필요.
저장소: https://github.com/sdkparkforbi/gva-scenario-2025-2035

## 1. bvar_industry.py — 36산업 베이지안 VAR(미네소타)

In [ ]:
# -*- coding: utf-8 -*-
"""36개 산업 실질부가가치 증가율에 대한 Large Bayesian VAR (켤레 미네소타 prior).

- 데이터 : gdp_industry_long.csv 의 실질부가가치(연쇄, 십억원), 1980Q1~2025Q4
- 변수   : g_t = 100 * Δlog(real VA)  (36개 산업, 정상시계열)
- 모형   : VAR(p=4),  y_t = c + B1 y_{t-1} + ... + B4 y_{t-4} + u_t,  u_t~N(0,Σ)
- prior  : 켤레 정규-역위샤트(Minnesota).  자기 1차항 prior평균 δ=0 (white-noise),
           lag l·변수 j 계수 분산 ∝ λ²/(l² σ_j²),  Σ~IW(diag(σ_i²), n+2)
- λ 선택 : 주변우도(marginal likelihood) 격자 최대화 (Giannone-Lenza-Primiceri 2015)

산출:
  bvar_coefficients.csv  : 사후평균 계수 B (k x n)
  bvar_sigma.csv         : 잔차 공분산 Σ
  bvar_summary.txt       : 선택 λ, 안정성, 적합도 등 진단
"""
import numpy as np
import pandas as pd
from scipy.special import multigammaln

P = 4                       # 시차 (요청: max 4)
DELTA = 0.0                 # 자기 1차항 prior 평균 (증가율→0, white-noise)
LAMBDAS = [0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0, 2.0, 5.0]


def load_growth():
    df = pd.read_csv("gdp_industry_long.csv", encoding="utf-8-sig")
    w = df.pivot(index="분기", columns="산업명", values="실질_십억원")
    order = sorted(w.index, key=lambda q: (int(q[:4]), int(q[5])))
    w = w.loc[order]
    g = 100.0 * np.log(w).diff().dropna()        # 100*Δlog, 183 x 36
    # 정유 등 극단 분기변동(예: 2020Q1 코크스·석유정제 -128%) 윈저화 → Σ·상수 왜곡 방지.
    # 실질부가가치=거대 산출−거대 투입이라 정유의 차액(부가가치)이 작고 변동이 극심하다.
    g = g.clip(-30, 30)
    return g


def build_xy(g, p):
    """y_t 와 x_t=(y_{t-1}..y_{t-p},1) 구성."""
    Y = g.values[p:]                              # (T x n)
    T, n = Y.shape
    X = np.ones((T, n * p + 1))
    for l in range(1, p + 1):
        X[:, (l - 1) * n:(l) * n] = g.values[p - l:-l]
    X[:, -1] = 1.0                                # 절편 (마지막 열)
    return Y, X


def ar_sigma(g, p):
    """각 변수 단변량 AR(p) 잔차표준편차 σ_i."""
    sig = np.zeros(g.shape[1])
    for i, col in enumerate(g.columns):
        y = g[col].values
        Y = y[p:]
        Z = np.column_stack([y[p - l:-l] for l in range(1, p + 1)] + [np.ones(len(Y))])
        beta, *_ = np.linalg.lstsq(Z, Y, rcond=None)
        sig[i] = (Y - Z @ beta).std(ddof=Z.shape[1])
    return sig


def minnesota_prior(n, p, sigma, lam):
    """켤레 미네소타 prior: B0(k x n), Omega0(k x k diag), Psi(n x n), d."""
    k = n * p + 1
    B0 = np.zeros((k, n))
    for i in range(n):
        B0[i, i] = DELTA                          # 자기 1차항
    om = np.zeros(k)
    for l in range(1, p + 1):
        for j in range(n):
            om[(l - 1) * n + j] = (lam ** 2) / (l ** 2 * sigma[j] ** 2)
    om[-1] = 1e6                                   # 절편: 매우 느슨
    Omega0 = np.diag(om)
    Psi = np.diag(sigma ** 2)
    d = n + 2                                      # E[Σ]=Psi/(d-n-1)=diag(σ²)
    return B0, Omega0, Psi, d


def posterior(Y, X, B0, Omega0, Psi, d):
    n = Y.shape[1]
    iO0 = np.diag(1.0 / np.diag(Omega0))
    iOb = iO0 + X.T @ X
    Ob = np.linalg.inv(iOb)
    Bb = Ob @ (iO0 @ B0 + X.T @ Y)
    Psib = Psi + Y.T @ Y + B0.T @ iO0 @ B0 - Bb.T @ iOb @ Bb
    Psib = (Psib + Psib.T) / 2
    db = d + Y.shape[0]
    return Bb, Ob, Psib, db, Omega0


def log_ml(Y, X, B0, Omega0, Psi, d):
    """주변우도 log p(Y) — 켤레 N-IW 정확식."""
    n, T = Y.shape[1], Y.shape[0]
    Bb, Ob, Psib, db, _ = posterior(Y, X, B0, Omega0, Psi, d)
    _, ld_O0 = np.linalg.slogdet(Omega0)
    _, ld_Ob = np.linalg.slogdet(Ob)
    _, ld_Psi = np.linalg.slogdet(Psi)
    _, ld_Psib = np.linalg.slogdet(Psib)
    return (-(n * T / 2) * np.log(np.pi)
            + (n / 2) * (ld_Ob - ld_O0)
            + (d / 2) * ld_Psi - (db / 2) * ld_Psib
            + multigammaln(db / 2, n) - multigammaln(d / 2, n))


def companion_max_eig(Bb, n, p):
    """동반행렬 최대 고유값 절댓값 (안정성: <1 이면 정상)."""
    A = Bb[:n * p, :].T                            # (n x n*p), 절편 제외
    comp = np.zeros((n * p, n * p))
    comp[:n, :] = A
    comp[n:, :-n] = np.eye(n * (p - 1))
    return np.max(np.abs(np.linalg.eigvals(comp)))


def main():
    g = load_growth()
    n = g.shape[1]
    Y, X = build_xy(g, P)
    sigma = ar_sigma(g, P)
    T = Y.shape[0]

    # λ 선택 (주변우도 최대)
    grid = []
    for lam in LAMBDAS:
        B0, Om0, Psi, d = minnesota_prior(n, P, sigma, lam)
        grid.append((lam, log_ml(Y, X, B0, Om0, Psi, d)))
    lam_opt = max(grid, key=lambda t: t[1])[0]

    # 최적 λ 추정
    B0, Om0, Psi, d = minnesota_prior(n, P, sigma, lam_opt)
    Bb, Ob, Psib, db, _ = posterior(Y, X, B0, Om0, Psi, d)
    Sigma = Psib / (db - n - 1)                    # 사후평균 Σ
    resid = Y - X @ Bb
    eig = companion_max_eig(Bb, n, P)

    # 적합도 (변수별 in-sample R²)
    r2 = pd.Series(1 - resid.var(0) / Y.var(0), index=g.columns)

    # 저장
    cols = ([f"L{l}.{c}" for l in range(1, P + 1) for c in g.columns] + ["const"])
    pd.DataFrame(Bb, index=cols, columns=g.columns).to_csv(
        "bvar_coefficients.csv", encoding="utf-8-sig")
    pd.DataFrame(Sigma, index=g.columns, columns=g.columns).to_csv(
        "bvar_sigma.csv", encoding="utf-8-sig")

    with open("bvar_summary.txt", "w", encoding="utf-8") as f:
        f.write("Large Bayesian VAR — 36개 산업 실질부가가치 증가율 (100*Δlog)\n")
        f.write(f"표본: {g.index[P]} ~ {g.index[-1]}  (유효관측 T={T}, 변수 n={n}, 시차 p={P})\n")
        f.write(f"prior: 켤레 미네소타, δ={DELTA}(white-noise), Σ~IW(diag(σ²), n+2)\n\n")
        f.write("[λ 선택: 주변우도]\n")
        for lam, ml in grid:
            mark = "  <== 선택" if lam == lam_opt else ""
            f.write(f"  λ={lam:>5}:  logML={ml:14.2f}{mark}\n")
        f.write(f"\n선택 λ* = {lam_opt}\n")
        f.write(f"안정성: 동반행렬 최대고유값 |z|max = {eig:.4f}  "
                f"({'정상(안정)' if eig < 1 else '비정상'})\n")
        f.write(f"평균 in-sample R² = {r2.mean():.3f}  "
                f"(min {r2.min():.3f}, max {r2.max():.3f})\n\n")
        f.write("[변수별 in-sample R²]\n")
        for c, v in r2.sort_values(ascending=False).items():
            f.write(f"  {v:5.3f}  {c}\n")

    print(f"BVAR 추정 완료: λ*={lam_opt}, |z|max={eig:.4f}, 평균R²={r2.mean():.3f}, T={T}, n={n}")
    print("저장: bvar_coefficients.csv, bvar_sigma.csv, bvar_summary.txt")


if __name__ == "__main__":
    main()


## 2. bvar_analysis.py — 연결성·GIRF·FEVD

In [ ]:
# -*- coding: utf-8 -*-
"""BVAR(4) 종합 분석 — 계수·충격반응(GIRF)·일반화FEVD·Diebold-Yilmaz 연결성.

bvar_industry.py 의 켤레 미네소타 BVAR(λ*=0.1) 사후분포에서 직접 추출하여
모든 통계량에 신용구간(credible interval)을 부여한다. 식별은 순서불변(GIRF/GFEVD).

산출:
  bvar_coef_persistence.csv  : 산업별 자기지속성(자기 lag합) 사후평균·신용구간
  bvar_top_spillovers.csv    : 강한 교차산업 스필오버(계수) 상위
  bvar_connectedness.csv     : 산업별 FROM/TO/NET 연결성(GFEVD 기반) + 총연결성
  bvar_girf_<shock>.csv       : 선택 산업 충격에 대한 GIRF(전 산업 반응, 신용구간)
  bvar_analysis_summary.txt
"""
import numpy as np
import pandas as pd
from scipy.stats import invwishart

from bvar_industry import load_growth, build_xy, ar_sigma, minnesota_prior, posterior, P

LAM = 0.1            # bvar_industry.py 에서 주변우도로 선택된 값
NDRAW = 600          # 사후 추출 수
H_IRF = 20           # 충격반응 시계(분기)
H_FEVD = 12          # FEVD/연결성 시계(3년)
SEED = 20260621
SHOCKS = ["컴퓨터, 전자 및 광학기기 제조업", "금융 및 보험업"]


def ma_theta(Bb, n, p, H):
    """MA(∞) 계수 Θ_0..Θ_H (각 n x n). 동반행렬 거듭제곱."""
    A = Bb[:n * p, :].T                       # [A1|A2|A3|A4]  (n x np)
    comp = np.zeros((n * p, n * p))
    comp[:n, :] = A
    if p > 1:
        comp[n:, :-n] = np.eye(n * (p - 1))
    J = np.zeros((n, n * p)); J[:n, :n] = np.eye(n)
    Th, Apow = [np.eye(n)], np.eye(n * p)
    for _ in range(H):
        Apow = comp @ Apow
        Th.append(J @ Apow @ J.T)
    return Th                                  # 길이 H+1


def gfevd(Th, Sigma, H):
    """일반화 FEVD (Pesaran-Shin), 행 정규화. θ̃[i,j] = i의 예측오차분산 중 j 기여분."""
    n = Sigma.shape[0]
    M = [Th[h] @ Sigma for h in range(H + 1)]   # Θ_h Σ
    sig = np.diag(Sigma)
    num = np.zeros((n, n)); den = np.zeros(n)
    for h in range(H + 1):
        num += M[h] ** 2
        den += np.einsum("ik,ik->i", M[h], Th[h])   # diag(Θ_h Σ Θ_h')
    theta = (num / sig[None, :]) / den[:, None]
    return theta / theta.sum(1, keepdims=True)       # 행합=1로 정규화


def connectedness(theta):
    """Diebold-Yilmaz: FROM(받는), TO(주는), NET, 총연결성 TCI(%)."""
    n = theta.shape[0]
    frm = 100 * (1 - np.diag(theta))                 # 행 비대각 합
    to = 100 * (theta.sum(0) - np.diag(theta))        # 열 비대각 합
    net = to - frm
    tci = frm.mean()
    return frm, to, net, tci


def girf(Th, Sigma, j, H):
    """변수 j 에 1 표준편차 일반화충격 → 전 변수 반응 (H+1 x n)."""
    sj = np.sqrt(Sigma[j, j])
    return np.array([(Th[h] @ Sigma[:, j]) / sj for h in range(H + 1)])


def main():
    rng = np.random.default_rng(SEED)
    g = load_growth(); names = list(g.columns); n = len(names)
    Y, X = build_xy(g, P); sigma = ar_sigma(g, P)
    B0, Om0, Psi, d = minnesota_prior(n, P, sigma, LAM)
    Bb, Ob, Psib, db, _ = posterior(Y, X, B0, Om0, Psi, d)

    Lu = np.linalg.cholesky(Ob)                       # row cov
    idx = {nm: i for i, nm in enumerate(names)}

    # 누적 저장소
    persist = np.zeros((NDRAW, n))                     # 자기 lag합
    A1diag = np.zeros((NDRAW, n))                      # 자기 1차항
    FROM = np.zeros((NDRAW, n)); TO = np.zeros((NDRAW, n))
    NET = np.zeros((NDRAW, n)); TCI = np.zeros(NDRAW)
    A1_sum = np.zeros((n, n))                          # 1차 계수 사후평균(스필오버용)
    girf_store = {s: np.zeros((NDRAW, H_IRF + 1, n)) for s in SHOCKS}
    n_stable = 0

    for s in range(NDRAW):
        Sig = invwishart.rvs(df=db, scale=Psib, random_state=rng)
        Lv = np.linalg.cholesky(Sig)
        Bs = Bb + Lu @ rng.standard_normal((Bb.shape[0], n)) @ Lv.T

        A_l = [Bs[(l) * n:(l + 1) * n, :].T for l in range(P)]   # A_{l+1}[c,j]
        persist[s] = sum(np.diag(Al) for Al in A_l)
        A1diag[s] = np.diag(A_l[0])
        A1_sum += A_l[0]

        Th = ma_theta(Bs, n, P, max(H_IRF, H_FEVD))
        # 안정성
        comp_eig = np.abs(np.linalg.eigvals(
            np.block([[Bs[:n * P, :].T],
                      [np.eye(n * (P - 1)), np.zeros((n * (P - 1), n))]])
            if P > 1 else Bs[:n, :].T)).max()
        n_stable += comp_eig < 1

        theta = gfevd(Th, Sig, H_FEVD)
        FROM[s], TO[s], NET[s], TCI[s] = connectedness(theta)
        for sh in SHOCKS:
            girf_store[sh][s] = girf(Th, Sig, idx[sh], H_IRF)

    A1_mean = A1_sum / NDRAW
    qs = lambda a, ax=0: np.percentile(a, [5, 16, 50, 84, 95], axis=ax)

    # 1) 자기지속성
    pq = qs(persist)
    pd.DataFrame({"산업": names, "지속성_p50": pq[2], "p05": pq[0], "p16": pq[1],
                  "p84": pq[3], "p95": pq[4]}).sort_values(
        "지속성_p50", ascending=False).to_csv(
        "bvar_coef_persistence.csv", index=False, encoding="utf-8-sig")

    # 2) 강한 스필오버 (1차 계수, 비대각, |평균|/사후표준편차 기준)
    A1_draws = None  # 메모리 절약: 표준편차는 평균제곱 누적 대신 근사 생략, 평균크기로 선정
    sp = []
    for c in range(n):
        for j in range(n):
            if c != j:
                sp.append((names[j], names[c], A1_mean[c, j]))
    sp = pd.DataFrame(sp, columns=["from_충격산업", "to_반응산업", "A1계수"])
    sp["abs"] = sp["A1계수"].abs()
    sp.sort_values("abs", ascending=False).head(25).drop(columns="abs").to_csv(
        "bvar_top_spillovers.csv", index=False, encoding="utf-8-sig")

    # 3) 연결성
    fq, tq, nq = qs(FROM), qs(TO), qs(NET)
    conn = pd.DataFrame({"산업": names,
                         "FROM_받는": fq[2], "TO_주는": tq[2],
                         "NET": nq[2], "NET_p05": nq[0], "NET_p95": nq[4]})
    conn = conn.sort_values("NET", ascending=False)
    conn.to_csv("bvar_connectedness.csv", index=False, encoding="utf-8-sig")

    # 4) GIRF 저장
    for sh in SHOCKS:
        gq = np.percentile(girf_store[sh], [5, 50, 95], axis=0)   # (3,H+1,n)
        out = {"h": np.arange(H_IRF + 1)}
        for i, nm in enumerate(names):
            out[f"{nm}_p50"] = gq[1, :, i]
            out[f"{nm}_p05"] = gq[0, :, i]
            out[f"{nm}_p95"] = gq[2, :, i]
        safe = sh.split(",")[0].split(" ")[0]
        pd.DataFrame(out).to_csv(f"bvar_girf_{safe}.csv", index=False, encoding="utf-8-sig")

    with open("bvar_analysis_summary.txt", "w", encoding="utf-8") as f:
        f.write("BVAR(4) 종합 분석 — 36개 산업 실질부가가치 증가율\n")
        f.write(f"사후추출 {NDRAW}회 (안정 draw 비율 {n_stable/NDRAW:.1%}), "
                f"GIRF/GFEVD 순서불변, FEVD 시계 H={H_FEVD}\n\n")
        f.write(f"[총연결성 TCI]  {TCI.mean():.1f}%  "
                f"(90% 신용구간 {np.percentile(TCI,5):.1f}~{np.percentile(TCI,95):.1f})\n")
        f.write("  → 전체 산업 예측오차분산의 이 비율이 '다른 산업'에서 전이됨\n\n")
        f.write("[순(NET) 연결성 상위 — 순(net) 충격 전파자(허브)]\n")
        for _, r in conn.head(8).iterrows():
            f.write(f"  {r['NET']:+6.1f}  {r['산업']}  (주는 {r['TO_주는']:.0f}, 받는 {r['FROM_받는']:.0f})\n")
        f.write("\n[순(NET) 연결성 하위 — 순 충격 수용자]\n")
        for _, r in conn.tail(6).iterrows():
            f.write(f"  {r['NET']:+6.1f}  {r['산업']}  (주는 {r['TO_주는']:.0f}, 받는 {r['FROM_받는']:.0f})\n")
        f.write("\n[자기지속성 상위/하위 (자기 lag 합)]\n")
        ps = pd.DataFrame({"산업": names, "p": pq[2]}).sort_values("p", ascending=False)
        for _, r in ps.head(5).iterrows():
            f.write(f"  {r['p']:+.2f}  {r['산업']}\n")
        f.write("  ...\n")
        for _, r in ps.tail(3).iterrows():
            f.write(f"  {r['p']:+.2f}  {r['산업']}\n")

    print(f"분석 완료: TCI={TCI.mean():.1f}%, 안정draw={n_stable/NDRAW:.1%}, draws={NDRAW}")
    print("저장: bvar_connectedness.csv, bvar_coef_persistence.csv, "
          "bvar_top_spillovers.csv, bvar_girf_*.csv, bvar_analysis_summary.txt")


if __name__ == "__main__":
    main()


## 3. bvarx_industry.py — 외생 결합 BVARX(위기더미·노동 구조제약)

In [ ]:
# -*- coding: utf-8 -*-
"""BVARX — 36개 산업 실질부가가치 증가율 + 외생 거시 블록 (시나리오 엔진).

  y_t = c + Σ_{l=1..p} B_l y_{t-l} + Σ_{s=0..q} Γ_s x_{t-s} + u_t,  u_t~N(0,Σ)
  내생 y : 36개 산업 100·Δlog(real VA)           (미네소타 prior, λ=0.1)
  외생 x : oil_g, trd_g, d_rrate, rfx_g, lab_g    (느슨한 prior, λx=1.0)
  내생 lag p=4, 외생 lag q=2(동시+2분기). 켤레 정규-역위샤트 사후평균.

외생 동태승수(dynamic multiplier) D_h = ∂y_{t+h}/∂x_t' 를 산출 →
각 외생충격(유가·환율·금리·교역·노동)의 산업별 전파 = 시나리오 시뮬레이션 기반.

산출: bvarx_coefficients.csv, bvarx_exog_multipliers.csv, bvarx_summary.txt,
      _bvarx_mult.json (시각화용)
"""
import json
import numpy as np
import pandas as pd

from bvar_industry import load_growth, ar_sigma

P, Q = 4, 2                 # 내생 lag, 외생 lag
LAM, LAMX = 0.1, 1.0        # 내생/외생 prior 수축강도
EXO = ["oil_g", "trd_g", "d_rrate", "rfx_g", "lab_g"]
# 노동 패스스루를 구조 탄력성(노동분배율)으로 제약. 축소형 계수(~1.4)는 경기 공행성을
# 반영해 구조적 인구감소에 적용 시 산출을 과대 위축시킨다 → 동시계수 prior평균=노동분배율.
# 동시계수 prior평균을 0.40으로 잡으면, 산업 간 동학 증폭 후 총량 가중 장기승수가
# 0.85(노동분배율 0.65 + 완만한 일반균형 승수) 수준이 되어 구조적으로 타당해진다.
LAB_SHARE, OM_LAB = 0.40, 0.0008
# 위기 더미: IMF·글로벌금융위기·코로나의 급락 분기를 더미로 흡수 → 상수가 평상시 동학 반영.
# 위기는 상시 상황이 아니므로 정상 표본에서 분리한다.
CRISIS = {"IMF": ["1998Q1", "1998Q2", "1998Q3"],
          "GFC": ["2008Q4", "2009Q1"],
          "COVID": ["2020Q1", "2020Q2"]}


def qkey(q):
    return (int(q[:4]), int(q[5]))


def build():
    g = load_growth()                                    # 36산업 증가율
    ex = pd.read_csv("exog_quarterly_filled.csv", encoding="utf-8-sig").set_index("분기")
    ex = ex[EXO].astype(float)
    common = sorted(set(g.index) & set(ex.index), key=qkey)
    g, ex = g.loc[common], ex.loc[common]
    n, ne = g.shape[1], ex.shape[1]
    T0 = max(P, Q)                                        # lag 확보 시작
    rows = list(range(T0, len(common)))
    Y = g.values[rows]                                   # (T x n)
    T = len(rows)
    # 설계: [내생 L1..Lp | 외생 L0..Lq | const]
    blocks = []
    for l in range(1, P + 1):
        blocks.append(g.values[[r - l for r in rows]])
    for s in range(0, Q + 1):
        blocks.append(ex.values[[r - s for r in rows]])
    qlabels = [common[r] for r in rows]
    dums = [np.array([1.0 if q in cq else 0.0 for q in qlabels]) for cq in CRISIS.values()]
    X = np.column_stack(blocks + dums + [np.ones(T)])   # [내생 | 외생 | 위기더미 | const]
    return g, ex, Y, X, n, ne, qlabels


def prior(n, ne, sig_y, sig_x):
    nd = len(CRISIS)
    k = n * P + ne * (Q + 1) + nd + 1
    B0 = np.zeros((k, n))                                # white-noise prior
    om = np.zeros(k)
    pos = 0
    for l in range(1, P + 1):                            # 내생: 미네소타
        for j in range(n):
            om[pos] = LAM ** 2 / (l ** 2 * sig_y[j] ** 2); pos += 1
    for s in range(0, Q + 1):                            # 외생: 느슨
        for m in range(ne):
            om[pos] = LAMX ** 2 / ((s + 1) ** 2 * sig_x[m] ** 2); pos += 1
    for _ in range(nd):                                  # 위기더미: 느슨(자유 추정)
        om[pos] = 1e6; pos += 1
    om[pos] = 1e6                                        # const
    # 노동계수 구조 제약: 동시계수 prior평균=노동분배율, 시차=0, 모두 tight
    lab = EXO.index("lab_g")
    for s in range(Q + 1):
        r = n * P + s * ne + lab
        B0[r, :] = LAB_SHARE if s == 0 else 0.0
        om[r] = OM_LAB
    return B0, np.diag(om), np.diag(sig_y ** 2), n + 2


def posterior(Y, X, B0, Om0, Psi, d):
    iO0 = np.diag(1 / np.diag(Om0))
    iOb = iO0 + X.T @ X
    Ob = np.linalg.inv(iOb)
    Bb = Ob @ (iO0 @ B0 + X.T @ Y)
    Psib = Psi + Y.T @ Y + B0.T @ iO0 @ B0 - Bb.T @ iOb @ Bb
    return Bb, (Psib + Psib.T) / 2, d + Y.shape[0]


def multipliers(Bb, n, ne, H=20):
    """외생 동태승수 D_h (n x ne), h=0..H.  D_h = Σ B_l D_{h-l} + Γ_h."""
    Bl = [Bb[(l - 1) * n:l * n, :].T for l in range(1, P + 1)]      # A_l (n x n)
    G = [Bb[n * P + s * ne: n * P + (s + 1) * ne, :].T for s in range(Q + 1)]  # Γ_s (n x ne)
    D = []
    for h in range(H + 1):
        Dh = G[h].copy() if h <= Q else np.zeros((n, ne))
        for l in range(1, P + 1):
            if h - l >= 0:
                Dh = Dh + Bl[l - 1] @ D[h - l]
        D.append(Dh)
    return D                                              # 길이 H+1, 각 (n x ne)


def main():
    g, ex, Y, X, n, ne, qs = build()
    sig_y = ar_sigma(g, P)
    sig_x = ex.std().values
    B0, Om0, Psi, d = prior(n, ne, sig_y, sig_x)
    Bb, Psib, db = posterior(Y, X, B0, Om0, Psi, d)
    Sigma = Psib / (db - n - 1)
    resid = Y - X @ Bb
    r2 = pd.Series(1 - resid.var(0) / Y.var(0), index=g.columns)

    # 안정성(내생 동반행렬)
    A = Bb[:n * P, :].T
    comp = np.zeros((n * P, n * P)); comp[:n, :] = A
    comp[n:, :-n] = np.eye(n * (P - 1))
    eig = np.abs(np.linalg.eigvals(comp)).max()

    # 외생 동태승수: 1 표준편차 충격에 대한 누적(레벨) 반응
    D = multipliers(Bb, n, ne, H=20)
    cum = {}
    for m, xname in enumerate(EXO):
        imp = np.array([D[h][:, m] * sig_x[m] for h in range(len(D))])   # (H+1 x n) 증가율 반응
        cum[xname] = np.cumsum(imp, axis=0)                              # 누적=레벨 반응

    # 저장: 20분기 누적 반응(장기 승수)
    longrun = pd.DataFrame({xname: cum[xname][-1] for xname in EXO}, index=g.columns)
    longrun.to_csv("bvarx_exog_multipliers.csv", encoding="utf-8-sig")
    cols = ([f"y.L{l}.{c}" for l in range(1, P + 1) for c in g.columns]
            + [f"x.L{s}.{m}" for s in range(Q + 1) for m in EXO] + ["const"])
    pd.DataFrame(Bb, index=cols, columns=g.columns).to_csv(
        "bvarx_coefficients.csv", encoding="utf-8-sig")

    # 시각화용: 주요 외생충격의 산업별 20분기 누적반응 + 시간경로(대표산업)
    viz = {"exo": EXO, "industries": list(g.columns),
           "longrun": {x: [round(v, 3) for v in longrun[x].values] for x in EXO},
           "sig_x": {x: round(float(s), 3) for x, s in zip(EXO, sig_x)}}
    json.dump(viz, open("_bvarx_mult.json", "w", encoding="utf-8"), ensure_ascii=False)

    with open("bvarx_summary.txt", "w", encoding="utf-8") as f:
        f.write("BVARX — 36산업 증가율 + 외생 5변수 (시나리오 엔진)\n")
        f.write(f"표본 {qs[0]}~{qs[-1]} (T={Y.shape[0]}), 내생 p={P}, 외생 q={Q}, "
                f"λ={LAM}, λx={LAMX}\n")
        f.write(f"안정성 |z|max={eig:.4f} ({'안정' if eig<1 else '불안정'}), "
                f"평균 R²={r2.mean():.3f}\n\n")
        f.write("[외생충격 1σ → 20분기 누적(레벨) 반응: 산업 평균, %p]\n")
        for x in EXO:
            v = longrun[x]
            f.write(f"  {x:8} 1σ={sig_x[EXO.index(x)]:.2f}: 산업평균 {v.mean():+.2f}, "
                    f"최대 {v.idxmax()[:10]} {v.max():+.2f}, 최소 {v.idxmin()[:10]} {v.min():+.2f}\n")
    print(f"BVARX 추정: 표본 {qs[0]}~{qs[-1]} T={Y.shape[0]}, |z|max={eig:.4f}, 평균R²={r2.mean():.3f}")
    print("저장: bvarx_coefficients.csv, bvarx_exog_multipliers.csv, bvarx_summary.txt")


if __name__ == "__main__":
    main()


## 4. check_reconcile.py — 산업 합=GDP 정합성 검증

In [ ]:
# -*- coding: utf-8 -*-
"""산업 합 = 총부가가치 = GDP 일치 검증 (명목·실질, 1980Q1~2025Q4).

명목: 36개 산업 합 == 총부가가치(1200), + 순생산물세(1300) == GDP(1400)  → 정확히 일치 기대.
실질: 연쇄가중(2020 기준) 볼륨지수는 가법성이 없어 합 != 총부가가치 (연쇄불일치, chain residual).
       → 2020년 부근에서 0에 수렴, 멀어질수록 괴리 커지는지 확인.
"""
import requests
import pandas as pd

API_KEY = open("API_KEY_BOK.txt", encoding="utf-8").read().strip()
URL = "http://ecos.bok.or.kr/api/StatisticSearch/{k}/json/kr/1/30000/{t}/Q/1980Q1/2025Q4"
LEAF = {  # 36개 말단 산업 코드
    "1101", "1102", "110301", "110302", "110303", "110304", "110305", "110306",
    "110307", "110308", "110309", "110310", "110311", "110312", "110313",
    "110401", "110402", "110403", "11051", "11052", "11053", "11054",
    "110601", "110602", "1107", "1108", "1109", "11141", "11142",
    "111501", "111502", "1110", "1111", "1112", "11131", "11132",
}


def load(table):
    rows = requests.get(URL.format(k=API_KEY, t=table), timeout=120) \
        .json().get("StatisticSearch", {}).get("row", [])
    rec = {}
    for r in rows:
        v = r.get("DATA_VALUE")
        if v in (None, "", "-"):
            continue
        rec.setdefault(r["ITEM_CODE1"], {})[r["TIME"]] = float(v)
    return rec


def reconcile(table, label):
    rec = load(table)
    leaf_sum = pd.DataFrame({c: rec[c] for c in LEAF if c in rec}).sort_index().sum(axis=1)
    gva = pd.Series(rec.get("1200", {}))          # 총부가가치(기초가격)
    tax = pd.Series(rec.get("1300", {}))          # 순생산물세
    gdp = pd.Series(rec.get("1400", {}))          # GDP(시장가격)
    df = pd.DataFrame({"leaf_sum": leaf_sum, "GVA": gva, "tax": tax, "GDP": gdp}).dropna()
    df["합-GVA"] = df["leaf_sum"] - df["GVA"]
    df["GVA+세-GDP"] = df["GVA"] + df["tax"] - df["GDP"]
    df["합대비_GVA_괴리%"] = (df["합-GVA"] / df["GVA"] * 100)
    print(f"\n===== {label} ({table}) =====")
    print(f"  [산업합 - 총부가가치]  최대절대오차 = {df['합-GVA'].abs().max():.3f} 십억원")
    print(f"  [총부가가치+순생산물세 - GDP]  최대절대오차 = {df['GVA+세-GDP'].abs().max():.4f} 십억원")
    print(f"  연쇄불일치(합/GVA 괴리%)  2020 부근 vs 양끝:")
    for q in ["1980Q1", "2000Q1", "2019Q4", "2020Q1", "2020Q4", "2025Q4"]:
        if q in df.index:
            print(f"      {q}: {df.loc[q, '합대비_GVA_괴리%']:+.3f}%")
    return df


if __name__ == "__main__":
    reconcile("200Y103", "명목")
    reconcile("200Y104", "실질(연쇄, 2020=100)")
